# 📈 Trabalho vs Desempenho - Análise Estatística
## Testes de Hipótese e Correlações

**Notebook 4/7** - Série: Trabalho Estudantil e Desempenho no ENEM

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import ttest_ind, mannwhitneyu, kruskal, spearmanr
from pathlib import Path

PROJECT_ROOT = Path('/home/interas/faculdade/ciencia-dados/enem-data-exploration')
DATA_FILE = PROJECT_ROOT / 'data' / 'processed' / 'enem_2023_trabalho_estudantil.parquet'
FIGURES_DIR = PROJECT_ROOT / 'reports' / 'figures' / 'unidade-3'

df = pd.read_parquet(DATA_FILE)
print(f"✅ Dados carregados: {len(df):,} registros")

## 1️⃣ Teste T: Trabalha vs Não Trabalha

In [ ]:
# Separar grupos
nao_trabalha = df[df['TRABALHA'] == 0]['NOTA_MEDIA_5'].dropna()
trabalha = df[df['TRABALHA'] == 1]['NOTA_MEDIA_5'].dropna()

# Teste T
t_stat, p_value = ttest_ind(nao_trabalha, trabalha)

print("📊 TESTE T - TRABALHA vs NÃO TRABALHA")
print("=" * 60)
print(f"Estatística t: {t_stat:.4f}")
print(f"P-valor: {p_value:.10f}")
print(f"\nMédia (não trabalha): {nao_trabalha.mean():.2f}")
print(f"Média (trabalha): {trabalha.mean():.2f}")
print(f"Diferença: {nao_trabalha.mean() - trabalha.mean():.2f} pontos")

# Cohen's d (tamanho do efeito)
pooled_std = np.sqrt(((len(nao_trabalha)-1)*nao_trabalha.std()**2 + (len(trabalha)-1)*trabalha.std()**2) / 
                     (len(nao_trabalha) + len(trabalha) - 2))
cohens_d = (nao_trabalha.mean() - trabalha.mean()) / pooled_std
print(f"\nCohen's d: {cohens_d:.4f}")

if abs(cohens_d) < 0.2:
    interpretacao = "pequeno"
elif abs(cohens_d) < 0.5:
    interpretacao = "médio"
else:
    interpretacao = "grande"
print(f"Tamanho do efeito: {interpretacao}")

if p_value < 0.001:
    print(f"\n✅ RESULTADO: Diferença ESTATISTICAMENTE SIGNIFICATIVA (p < 0.001)")
else:
    print(f"\n⚠️ P-valor: {p_value:.4f}")

## 2️⃣ ANOVA: Situação de Trabalho (4 grupos)

In [ ]:
# Preparar grupos
grupos = []
labels = []
for cat in df['Q007_label'].unique():
    if pd.notna(cat):
        grupo_data = df[df['Q007_label'] == cat]['NOTA_MEDIA_5'].dropna()
        if len(grupo_data) > 0:
            grupos.append(grupo_data)
            labels.append(cat)

# ANOVA
f_stat, p_value_anova = stats.f_oneway(*grupos)

print("\n📊 ANOVA - SITUAÇÃO DE TRABALHO (4 GRUPOS)")
print("=" * 60)
print(f"Estatística F: {f_stat:.4f}")
print(f"P-valor: {p_value_anova:.10f}")

if p_value_anova < 0.001:
    print(f"\n✅ RESULTADO: Há diferenças significativas entre os grupos (p < 0.001)")

# Médias por grupo
print("\nMédias por grupo:")
for label, grupo in zip(labels, grupos):
    print(f"  {label}: {grupo.mean():.2f}")

## 3️⃣ Correlação: Carga Horária × Nota

In [ ]:
# Correlação de Spearman (adequada para ordinais)
dados_completos = df[['Q008_ord', 'NOTA_MEDIA_5']].dropna()
rho, p_value_corr = spearmanr(dados_completos['Q008_ord'], dados_completos['NOTA_MEDIA_5'])

print("\n📊 CORRELAÇÃO DE SPEARMAN")
print("=" * 60)
print(f"Carga Horária × Nota Média")
print(f"\nrho (Spearman): {rho:.4f}")
print(f"P-valor: {p_value_corr:.10f}")

if abs(rho) < 0.3:
    forca = "fraca"
elif abs(rho) < 0.7:
    forca = "moderada"
else:
    forca = "forte"

direcao = "negativa" if rho < 0 else "positiva"
print(f"\nInterpretação: Correlação {forca} {direcao}")

if p_value_corr < 0.001:
    print(f"✅ Estatisticamente significativa (p < 0.001)")

In [ ]:
# Scatter plot com linha de tendência
plt.figure(figsize=(12, 7))

# Calcular médias por carga horária
medias_carga = df.groupby('Q008_ord')['NOTA_MEDIA_5'].agg(['mean', 'count']).reset_index()

plt.scatter(medias_carga['Q008_ord'], medias_carga['mean'], 
           s=medias_carga['count']/100, alpha=0.6, c=medias_carga['mean'],
           cmap='RdYlGn', edgecolors='black', linewidth=2)

# Linha de tendência
z = np.polyfit(medias_carga['Q008_ord'], medias_carga['mean'], 1)
p = np.poly1d(z)
plt.plot(medias_carga['Q008_ord'], p(medias_carga['Q008_ord']), 
        "r--", linewidth=2, label=f'Tendência (rho={rho:.3f})')

plt.xlabel('Carga Horária (ordinal: 0=nenhuma, 5=40h+)', fontsize=12, fontweight='bold')
plt.ylabel('Nota Média ENEM', fontsize=12, fontweight='bold')
plt.title('Relação: Carga Horária × Desempenho', fontsize=14, fontweight='bold')
plt.colorbar(label='Nota Média')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '07_correlacao_carga_horaria.png', dpi=300, bbox_inches='tight')
plt.show()

## 4️⃣ Análise por Disciplina

In [ ]:
# Gap por disciplina
disciplinas = ['NU_NOTA_MT', 'NU_NOTA_LC', 'NU_NOTA_CH', 'NU_NOTA_CN', 'NU_NOTA_REDACAO']
nomes = ['Matemática', 'Linguagens', 'Humanas', 'Natureza', 'Redação']

gaps = []
for disc in disciplinas:
    if disc in df.columns:
        media_nao = df[df['TRABALHA'] == 0][disc].mean()
        media_sim = df[df['TRABALHA'] == 1][disc].mean()
        gap = media_nao - media_sim
        gaps.append(gap)
    else:
        gaps.append(0)

# Visualizar
plt.figure(figsize=(12, 6))
colors = ['#1976D2' if g > 0 else '#D32F2F' for g in gaps]
bars = plt.bar(nomes, gaps, color=colors, alpha=0.7, edgecolor='black')
plt.axhline(y=0, color='black', linestyle='-', linewidth=0.8)
plt.ylabel('Gap de Desempenho (pontos)', fontsize=12, fontweight='bold')
plt.title('Impacto do Trabalho por Disciplina\n(Não trabalha - Trabalha)', 
         fontsize=14, fontweight='bold')
plt.grid(axis='y', alpha=0.3)

for bar, gap in zip(bars, gaps):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
            f'{gap:.1f}',
            ha='center', va='bottom' if gap > 0 else 'top',
            fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig(FIGURES_DIR / '08_gap_por_disciplina.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n📊 GAP POR DISCIPLINA:")
for nome, gap in zip(nomes, gaps):
    print(f"  {nome}: {gap:.2f} pontos")

## ✅ Síntese dos Testes

### Principais Resultados:
1. **Teste T:** Diferença significativa entre quem trabalha e quem não trabalha
2. **ANOVA:** Diferenças significativas entre os 4 grupos de trabalho
3. **Correlação:** Relação negativa entre carga horária e desempenho
4. **Por Disciplina:** Impacto presente em todas as áreas do conhecimento

➡️ **Próximo:** `05_interseccoes_trabalho.ipynb` - Análise de subgrupos